# FleetIQ / SmartTaxi — 07. Advanced Tabular Modeling & Champion Selection

## Objective
Train, benchmark, and compare 5 tabular regression models on the exact same chronologically partitioned dataset:
1. **Baseline** (Historical Average Speed)
2. **Random Forest** (Non-linear Bagging Regressor)
3. **LightGBM** (High-throughput Gradient Boosting)
4. **XGBoost** (Extreme Gradient Boosting)
5. **CatBoost** (Ordered Boosting Regressor)

Identify the Champion model and serialize the production bundle for microservice deployment.


In [1]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from src.models.train import train_and_evaluate_models

# Execute tabular training across all candidates
summary = train_and_evaluate_models()
print('Champion selected:', summary['champion_model'])


Loading processed datasets...
Dataset shape: Train=(138718, 9), Val=(29725, 9), Test=(29726, 9)
Features utilized: ['distance_haversine_km', 'distance_manhattan_km', 'pickup_hour', 'pickup_dayofweek', 'is_weekend', 'is_rush_hour', 'sin_hour', 'cos_hour', 'passenger_count']

--- [1/5] Fitting Historical Average Speed Baseline ---
Baseline Test MAE: 7.907 min, R2: 0.2584

--- [2/5] Training Random Forest Regressor ---


Random Forest Test MAE: 5.202 min, R2: 0.6866

--- [3/5] Training XGBoost Regressor ---


XGBoost Test MAE: 5.295 min, R2: 0.6785

--- [4/5] Training LightGBM Regressor ---


LightGBM Test MAE: 5.28 min, R2: 0.6802

--- [5/5] Training CatBoost Regressor ---


CatBoost Test MAE: 5.402 min, R2: 0.6697

--- Performing Subgroup Slice Analysis on Test Set ---

 CHAMPION MODEL SELECTED: Random Forest
 Test MAE: 5.202 min (312.1 sec)
 Test RMSE: 8.54 min
 Test R2: 0.6866
 Single-row Latency: 16.418 ms
Saved Champion model bundle to C:\Users\user\OneDrive - ESPRIT\Bureau\smart-eta-prediction\smart-eta-prediction\models\champion_eta_model.joblib
Champion selected: Random Forest


## 1. Tableau Comparatif Complet du Benchmark (Test Set)
Évaluation comparative sur le jeu de test chronologique (29 726 courses réelles, fin octobre 2025).


In [2]:
models_summary = summary['models_summary']
df_res = pd.DataFrame(models_summary).T
df_res = df_res.sort_values(by='mae_minutes')
df_res[['mae_minutes', 'mae_seconds', 'rmse_minutes', 'r2', 'mape_pct', 'latency_ms']]


,mae_minutes,mae_seconds,rmse_minutes,r2,mape_pct,latency_ms
Random Forest,5.202,312.1,8.540,0.6866,32.07,16.418
LightGBM,5.280,316.8,8.626,0.6802,33.01,1.057
XGBoost,5.295,317.7,8.649,0.6785,33.20,1.806
CatBoost,5.402,324.1,8.767,0.6697,34.19,1.231
Baseline (Avg Speed),7.907,474.4,13.136,0.2584,41.95,0.021


## 2. Visualisation Graphique : MAE et Coefficient R²


In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Barplot MAE
colors_mae = ['#2B6CB0' if idx == summary['champion_model'] else '#CBD5E0' for idx in df_res.index]
df_res['mae_minutes'].plot(kind='bar', ax=ax1, color=colors_mae, edgecolor='black')
ax1.set_title('Test MAE (Minutes) — Plus bas est meilleur')
ax1.set_ylabel('MAE (min)')
ax1.set_xticklabels(df_res.index, rotation=30, ha='right')
ax1.grid(True, alpha=0.3)

# Barplot R2
colors_r2 = ['#2E8540' if idx == summary['champion_model'] else '#A0AEC0' for idx in df_res.index]
df_res['r2'].plot(kind='bar', ax=ax2, color=colors_r2, edgecolor='black')
ax2.set_title('Test R² Score — Plus haut est meilleur')
ax2.set_ylabel('R² Score')
ax2.set_xticklabels(df_res.index, rotation=30, ha='right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


<Figure size 1300x450 with 2 Axes>

## 3. Conclusion du Benchmark & Choix d'Architecture pour le Déploiement

### Analyse Comparative :
- **Le Champion de Précision : Random Forest**
  - **MAE minimale** : **5.20 minutes** (gain de **34%** par rapport aux 7.91 min de la baseline).
  - **Meilleur pouvoir explicatif** : $R^2 = 0.687$ (hausse de **+165%** par rapport au 0.258 de la baseline).
  - **Latence d'inférence** : **21.37 ms**, parfaitement conforme au SLA de production ($< 50$ ms).
- **L'Alternative Haut Débit : LightGBM**
  - Atteint un $R^2 = 0.680$ et une MAE de $5.28$ min avec une vitesse d'exécution record de **1.10 ms** (idéal pour absorber des pics de charge extrêmes à 1 000 requêtes/seconde).

### Application Concrète Côté Déploiement :
1. **Persistance du Modèle Champion** : Le modèle Random Forest a été sérialisé et enregistré sous `models/champion_eta_model.joblib` avec l'ensemble de ses 100 estimateurs et ses métadonnées.
2. **Intégration FastAPI (`app.py`)** : Le microservice charge ce bundle binaire en mémoire RAM dès son initialisation (`@asynccontextmanager lifespan`) pour répondre aux requêtes de courses de passagers (`POST /predict/trip-duration`).
3. **Dashboard de Supervision (`app_streamlit.py`)** : Ce tableau de benchmark est exporté dans `models/model_comparison_results.json` et alimente directement l'onglet 3 de l'interface Streamlit.
